# ch08 jieba 中文分词实战（练习版）

> 按提示补全 `____`，跑通所有 assert。真值全部来自实跑（jieba 0.42.1）。
>
> 本章依赖 `jieba`（已装）；**不下载任何模型**，断网可跑。
> `VOCAB` / `clean` / `fmm` / `bmm` / `tokenize` 已给好（与 ch01 同口径），
> 你只需要补 jieba 相关的部分。挖空块内每条语句保持**单行**，控制流都在块外。

In [ ]:
import math
import re
import tempfile
import time
from collections import Counter
from pathlib import Path

import jieba
import jieba.analyse
import jieba.posseg as pseg
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

jieba.setLogLevel(60)          # 关掉 "Building prefix dict from ..." 的日志
DATA = Path("data")
CSV = DATA / "synth_text.csv"

In [ ]:
df = pd.read_csv(CSV, keep_default_na=False)
jieba.initialize()             # 惰性初始化：不先调一次，jieba.dt.total 会是 0

# ---- 以下词表 / 清洗 / 分词函数与 ch01 完全一致（本章要做对照，必须同口径）----
VOCAB = [
    "变压器", "断路器", "隔离开关", "避雷器", "电流互感器", "绝缘子", "套管", "母线",
    "渗油", "漏油", "油迹", "油污", "滴油", "油位下降",
    "锈蚀", "生锈", "铁锈", "漆面剥落", "镀锌层脱落",
    "破损", "裂纹", "碎裂", "缺口", "掉瓷", "箱体变形",
    "发热", "温度偏高", "过热", "烫手", "温升异常", "过热保护",
    "异物", "鸟巢", "塑料袋", "树枝", "风筝线搭挂",
    "发现", "巡检", "位于", "存在", "缺陷", "一般", "严重", "危急",
    "接报", "详见", "保护", "动作",
    "请安排抢修", "请尽快核实", "已通知运维班", "请列入本周计划",
    "需停电处理", "请核实后反馈",
    "研究", "研究生", "生命", "命", "起源",
]
VOCAB_SET = set(VOCAB)
MAX_WORD_LEN = max(len(w) for w in VOCAB)
SEG_RE = re.compile(r"[A-Za-z0-9_]+|[\u4e00-\u9fff]+|[^\sA-Za-z0-9_\u4e00-\u9fff]")
HAN_RE = re.compile(r"[\u4e00-\u9fff]+")


def clean(text: str) -> str:
    """ch01 的五步清洗管道（去 HTML/URL、标点归一、去汉字间空格）。"""
    s = re.sub(r"<[^>]+>", "", text)
    s = re.sub(r"https?://\S+", "", s)
    s = s.replace(",", "，")
    s = re.sub(r"([，。！？])\1+", r"\1", s)
    s = re.sub(r"(?<=[\u4e00-\u9fff])\s+(?=[\u4e00-\u9fff])", "", s)
    return re.sub(r"\s+", " ", s).strip()


def fmm(seg: str) -> list[str]:
    """正向最大匹配（ch01 §1.6）。"""
    out, i = [], 0
    while i < len(seg):
        length = min(MAX_WORD_LEN, len(seg) - i)
        while length > 1 and seg[i:i + length] not in VOCAB_SET:
            length -= 1
        out.append(seg[i:i + length])
        i += length
    return out


def bmm(seg: str) -> list[str]:
    """逆向最大匹配（ch01 §1.7）。"""
    out, i = [], len(seg)
    while i > 0:
        length = min(MAX_WORD_LEN, i)
        while length > 1 and seg[i - length:i] not in VOCAB_SET:
            length -= 1
        out.insert(0, seg[i - length:i])
        i -= length
    return out


def tokenize(text: str) -> list[str]:
    """两级切分：纯汉字段走 fmm，英文/数字/标点整体入列（ch01 §1.8）。"""
    out = []
    for seg in SEG_RE.findall(text):
        out.extend(fmm(seg) if HAN_RE.fullmatch(seg) else [seg])
    return out


desc_clean = df["缺陷描述"].map(clean)
BAD = {"", "N/A", "-", "无", "''", '""'}
keep_idx = [i for i, t in enumerate(desc_clean) if t.strip() not in BAD]
texts = [desc_clean[i] for i in keep_idx]
labels = df["缺陷类型"].to_numpy()[keep_idx]

print("原始 %d 条 | 清洗后可用 %d 条" % (len(df), len(texts)))
print("手写词表 %d 个词 | 最长 %d 字" % (len(VOCAB), MAX_WORD_LEN))
print("首条：", texts[0])

## 任务 1：三方对照（FMM / BMM / jieba）

同一句话切三遍。注意 `电流互感器发热` 这一行 —— **手写对了、jieba 错了**，
因为 jieba 的通用词典里没有这个电力复合词。

In [ ]:
CASES = ["研究生命起源", "变压器渗油", "电流互感器发热", "3号主变渗漏油", "隔离开关破损"]

print("%-10s %-20s %-20s %s" % ("输入", "正向最大匹配", "逆向最大匹配", "jieba"))
for t in CASES:
    a, b, c = fmm(t), bmm(t), jieba.lcut(t)
    print("%-10s %-20s %-20s %s" % (t, "/".join(a), "/".join(b), "/".join(c)))

print()
print("三种切法两两都不完全一致 —— 它们是三套不同的「什么算一个词」的定义。")

# ---- 验收 ----
assert fmm("研究生命起源") == ["研究生", "命", "起源"], "正向贪心：先把「研究生」吃掉"
assert bmm("研究生命起源") == ["研究", "生命", "起源"], "逆向贪心：从右往左吃到「生命」"
assert jieba.lcut("研究生命起源") == ["研究", "生命", "起源"]
assert fmm("电流互感器发热") == ["电流互感器", "发热"], "手写词表里有「电流互感器」这个长词"
assert jieba.lcut("电流互感器发热") == ["电流", "互感器", "发热"], "jieba 词典里没有「电流互感器」"
assert jieba.lcut("隔离开关破损") == ["隔离", "开关", "破损"], "jieba 词典里也没有「隔离开关」"

## 任务 2：jieba 三种模式

精确模式是**合法切分**（首尾相接拼回原句），全模式**不是**（词互相重叠）。

In [ ]:
MODE_CASES = ["主变压器渗漏油严重", "3号主变温升异常"]

for s in MODE_CASES:
    precise, full, search = jieba.lcut(s), jieba.lcut(s, cut_all=True), jieba.lcut_for_search(s)
    print(s)
    print("  精确模式   %d 个：" % len(precise), "/".join(precise))
    print("  全模式     %d 个：" % len(full), "/".join(full))
    print("  搜索引擎   %d 个：" % len(search), "/".join(search))
    print()

# ---- 验收 ----
assert jieba.lcut("主变压器渗漏油严重") == ["主", "变压器", "渗漏", "油", "严重"]
assert jieba.lcut("主变压器渗漏油严重", cut_all=True) == ["主", "变压", "变压器", "渗漏", "漏油", "严重"]
assert jieba.lcut_for_search("主变压器渗漏油严重") == ["主", "变压", "变压器", "渗漏", "油", "严重"]
assert len(jieba.lcut("主变压器渗漏油严重", cut_all=True)) == 6, "全模式会输出所有可能词，含重叠"
assert len(jieba.lcut("主变压器渗漏油严重")) == 5, "精确模式保证「切出来的词首尾相接、正好拼回原句」"
assert jieba.lcut("3号主变温升异常") == ["3", "号主", "变温", "升", "异常"], "jieba 也会切错：HMM 把「号主/变温」拼出来了"

## 任务 3：前缀词典的真实结构

`jieba.dt.FREQ` 把「词」和「词的所有前缀」放在同一张表里，**纯前缀记 freq=0**。
另外注意最后一行：`sum(FREQ.values())` 和 `jieba.dt.total` **不相等**。

In [ ]:
FREQ = jieba.dt.FREQ          # 词 + 所有前缀；纯前缀条目的 freq 人为记 0
TOTAL = jieba.dt.total        # DP 的归一化分母
LOG_TOTAL = math.log(TOTAL)

n_all = len(FREQ)
n_prefix_only = sum(1 for v in FREQ.values() if not v)
freq_sum = sum(FREQ.values())

print("FREQ 条目 %d | 纯前缀 %d | 真词 %d" % (n_all, n_prefix_only, n_all - n_prefix_only))
print("sum(FREQ.values()) = %d" % freq_sum)
print("jieba.dt.total     = %d   <- 分母必须是这个" % TOTAL)
print("两者差 %d —— 所以别自己重算 total（差这一点点，DP 的口径就和 jieba 不一样了）" % (TOTAL - freq_sum))
print()
print("%-14s %-8s %s" % ("片段", "in FREQ", "freq"))
for w in ["研", "研究", "研究生", "研究生命", "主", "主变", "温升", "发现", "AT", "STATION_E_02"]:
    print("%-14s %-8s %s" % (w, w in FREQ, FREQ.get(w)))

# ---- 验收 ----
assert n_all == 498113 and n_prefix_only == 149068
assert freq_sum == 60101964 and TOTAL == 60101967
assert FREQ["研究"] == 35029 and FREQ["研究生"] == 1816 and FREQ["命"] == 11603
assert FREQ["温升"] == 0, "「温升」在 FREQ 里但 freq=0 —— 它只是前缀索引（某个以「温升」开头的词的前缀），不是词"
assert "主变" not in FREQ, "「主变」连前缀都不是 —— 所以默认切分只能切出「主/变」"
assert "STATION_E_02" not in FREQ

## 任务 4：手写 DAG 构造

`while i < N and frag in FREQ` 已经在帮你往右扩了，你只需要**判定 + 收尾**。
关键：`freq==0` 的前缀不能收，但**还得继续往后扩**。

In [ ]:
def make_dag(s: str) -> dict:
    """构造 DAG：dag[k] = 以 k 为起点、所有「落在词典里」的终点。"""
    dag = {}
    N = len(s)
    for k in range(N):
        ends, i, frag = [], k, s[k]
        while i < N and frag in FREQ:
            ends.append(i) if FREQ[frag] else None
            i = i + 1
            frag = s[k:i + 1]
        if not ends:
            ends.append(k)      # 整段都不在词典里：单字成词
        dag[k] = ends
    return dag

In [ ]:
DEMO = "研究生命起源"

dag_demo = make_dag(DEMO)

for k in sorted(dag_demo):
    cand = [DEMO[k:x + 1] for x in dag_demo[k]]
    print("起点 %d「%s」-> 终点 %-12s 候选词 %s" % (k, DEMO[k], dag_demo[k], cand))

print()
print("注意起点 0：候选是「研 / 研究 / 研究生」——DAG 只给候选，不负责选哪个。")

# ---- 验收 ----
assert dag_demo == {0: [0, 1, 2], 1: [1], 2: [2, 3], 3: [3], 4: [4, 5], 5: [5]}
assert [DEMO[0:x + 1] for x in dag_demo[0]] == ["研", "研究", "研究生"]
assert [DEMO[2:x + 1] for x in dag_demo[2]] == ["生", "生命"]

## 任务 5：对数概率 DP

从后往前扫，每次在 `dag[idx]` 的所有出边里取「本词权重 + 后缀最优值」最大的一格。

In [ ]:
def wlog(word: str) -> float:
    """单个词的对数概率权重：log(freq) - log(total)。

    未登录词 freq 为 0 → `FREQ.get(word) or 1` 兜成 1，否则 math.log(0) 直接 ValueError。
    """
    return math.log(FREQ.get(word) or 1) - LOG_TOTAL

In [ ]:
def dp_route(s: str, dag: dict) -> dict:
    """从后往前 DP：route[k] = (从 k 到句尾的最优「对数概率之和」, 这一步切在哪)。"""
    N = len(s)
    route = {N: (0.0, 0)}          # 句尾哨兵：后面没词了，分值为 0
    for idx in range(N - 1, -1, -1):
        cand = [(wlog(s[idx:x + 1]) + route[x + 1][0], x) for x in dag[idx]]
        route[idx] = max(cand)
    return route

In [ ]:
def cut_dp(s: str) -> list[str]:
    """把 DP 的最优路径还原成切分结果。"""
    dag = make_dag(s)
    route = dp_route(s, dag)
    out, i, N = [], 0, len(s)
    while i < N:
        j = route[i][1]
        out.append(s[i:j + 1])
        i = j + 1
    return out


for s in ["研究生命起源", "电流互感器发热", "3号主变渗漏油"]:
    print("%-10s -> %s" % (s, "/".join(cut_dp(s))))

print()
print("「研究生命起源」的 route（起点 -> (分值, 切点)）：")
route_demo = dp_route("研究生命起源", make_dag("研究生命起源"))
for k in range(7):
    print("  %d %s" % (k, route_demo[k]))

print()
print("为什么 DP 能切对？它在所有合法路径里挑「对数概率之和最大」的那条：")
print("  「研/究/生/命/起/源」每个字都是低频，6 个 log 加起来很小；")
print("  「研究/生命/起源」三个高频词，3 个 log 加起来明显更大。")

# ---- 验收 ----
assert cut_dp("研究生命起源") == ["研究", "生命", "起源"]
assert cut_dp("电流互感器发热") == ["电流", "互感器", "发热"]
assert cut_dp("3号主变渗漏油") == ["3", "号", "主", "变", "渗漏", "油"], "纯 DP 拿 HMM 的未登录词没办法"

## 任务 6：复刻的最后一步 —— 英数字符合并

`merge_eng` 已经给好了（有控制流，不能放挖空块里）。
你只需要把 `cut_dp` 的结果交给它。

In [ ]:
RE_ENG = re.compile("[a-zA-Z0-9]")


def merge_eng(pieces: list[str]) -> list[str]:
    """把切分结果里「连续的单个英数字符」合并成一串。

    jieba 的 `__cut_DAG_NO_HMM` 在三模式之外还多做这一步：DP 只认汉字词，
    英文/数字必然是单字符 token，连着的要拼回去，否则 `STATION_E_02` 就成 12 个 token。
    """
    out, buf = [], ""
    for w in pieces:
        if RE_ENG.match(w) and len(w) == 1:
            buf += w
        else:
            if buf:
                out.append(buf)
                buf = ""
            out.append(w)
    if buf:
        out.append(buf)
    return out


def cut_dp_eng(s: str) -> list[str]:
    merged = merge_eng(cut_dp(s))
    return merged


demo_s = "STATION_E_02绝缘子渗油"
print("裸 DP    :", "/".join(cut_dp(demo_s)))
print("+英数合并 :", "/".join(cut_dp_eng(demo_s)))
print("jieba    :", "/".join(jieba.lcut(demo_s, HMM=False)))

# ---- 验收 ----
assert cut_dp(demo_s)[:3] == ["S", "T", "A"], "裸 DP 把编号逐字符切碎"
assert cut_dp_eng(demo_s)[0] == "STATION"
assert cut_dp_eng(demo_s) == jieba.lcut(demo_s, HMM=False)

## 任务 7：全语料对账

这一步会给你本章最震撼的数字：`0/570` → `570/570`。

In [ ]:
n_naive = sum(1 for t in texts if cut_dp(t) == jieba.lcut(t, HMM=False))
n_eng = sum(1 for t in texts if cut_dp_eng(t) == jieba.lcut(t, HMM=False))

print("裸 DP          vs jieba(HMM=False)：%d / %d = %.6f"
      % (n_naive, len(texts), n_naive / len(texts)))
print("DP + 英数合并   vs jieba(HMM=False)：%d / %d = %.6f"
      % (n_eng, len(texts), n_eng / len(texts)))
print()
print("首条：", texts[0])
print("  裸 DP    :", "/".join(cut_dp(texts[0])))
print("  DP+合并  :", "/".join(cut_dp_eng(texts[0])))
print("  jieba    :", "/".join(jieba.lcut(texts[0], HMM=False)))

# ---- 验收 ----
assert n_naive == 0, "少了「英数字符合并」这一步，570 条一条都对不上"
assert n_eng == len(texts) == 570, "补上这一步，就 100% 复刻了 jieba 的词典分词"

## 任务 8：HMM 开关（未登录词猜词）

In [ ]:
n_hmm = sum(1 for t in texts if jieba.lcut(t) != jieba.lcut(t, HMM=False))

print("HMM 开关造成差异：%d / %d = %.6f" % (n_hmm, len(texts), n_hmm / len(texts)))
print()
for t in ["3号主变渗漏油", "主变压器渗漏油严重", "未登录词阿巴阿巴"]:
    print("  %-14s HMM=True  %-24s HMM=False %s"
          % (t, "/".join(jieba.lcut(t)), "/".join(jieba.lcut(t, HMM=False))))
print()
print("HMM 在这批工单上**净收益是正的**（主变 / 渗漏油 这类电力词词典里都没有），")
print("但它也会犯错 —— 见 §8.4「3号主变温升异常」被切成「号主 / 变温」。")

# ---- 验收 ----
assert n_hmm == 186
assert jieba.lcut("3号主变渗漏油") == ["3", "号", "主变", "渗漏", "油"], "HMM 把未登录词「主变」拼出来了"
assert jieba.lcut("3号主变渗漏油", HMM=False) == ["3", "号", "主", "变", "渗漏", "油"]

## 任务 9：用户词典（电力术语）

按 `词 词频 [词性]` 格式写一份词典文件。词频是**优先级**，要给得够大。

In [ ]:
USER_DICT = Path(tempfile.gettempdir()) / "nlp08_user_dict.txt"

USER_DICT.write_text(
    "主变 100 n\n渗漏油 100 n\n母线巡检 100 n\n3号主变 100 n\n温升异常 100 n\n",
    encoding="utf-8",
)

print("用户词典：", USER_DICT)
print(USER_DICT.read_text(encoding="utf-8"))

# ---- 验收 ----
rows_ud = [ln.split() for ln in USER_DICT.read_text(encoding="utf-8").splitlines() if ln.strip()]
assert len(rows_ud) == 5
assert all(len(r) >= 2 for r in rows_ud), "每行至少要有「词 + 词频」"

## 任务 10：用独立 Tokenizer 加载词典

**不要**用 `jieba.load_userdict` 直接改全局 —— 后面还要用默认分词做对照。

In [ ]:
SENT1 = "母线巡检时发现异物，位于3号主变，严重渗漏油"
SENT2 = "3号主变温升异常"

tk = jieba.Tokenizer()
tk.initialize()
tk.load_userdict(str(USER_DICT))

for name, cut in [("全局默认", jieba.lcut), ("独立实例 + 用户词典", tk.lcut)]:
    print("%s：" % name)
    print("  ", "/".join(cut(SENT1)))
    print("  ", "/".join(cut(SENT2)))

# ---- 验收 ----
assert jieba.lcut(SENT1) == ["母线", "巡检", "时", "发现", "异物", "，",
                             "位于", "3", "号", "主变", "，", "严重", "渗漏", "油"]
assert tk.lcut(SENT1) == ["母线巡检", "时", "发现", "异物", "，",
                          "位于", "3号主变", "，", "严重", "渗漏油"]
assert jieba.lcut(SENT2) == ["3", "号主", "变温", "升", "异常"], "默认分词在 OOV 上切错了"
assert tk.lcut(SENT2) == ["3号主变", "温升异常"], "用户词典把错切成对"
assert jieba.lcut(SENT2) != tk.lcut(SENT2), "独立实例生效，且没有污染全局"

## 任务 11：词性标注 + 语料级词性分布

In [ ]:
SAMPLE = "母线巡检时发现严重渗漏油，位于3号主变，请尽快核实"

tagged = [(w, f) for w, f in pseg.cut(SAMPLE)]
pos_all = Counter(f for t in texts for w, f in pseg.cut(t))

print("词性标注：")
print("  " + " | ".join("%s/%s" % (w, f) for w, f in tagged))
print()
print("全语料词性种类 %d 种 | Top 8：%s" % (len(pos_all), pos_all.most_common(8)))
print("  n(名词)=%d  v(动词)=%d  x(标点)=%d  eng(英文)=%d  m(数词)=%d"
      % (pos_all["n"], pos_all["v"], pos_all["x"], pos_all["eng"], pos_all["m"]))

# ---- 验收 ----
tag_map = dict(tagged)
assert tag_map["严重"] == "a" and tag_map["位于"] == "v" and tag_map["尽快"] == "d"
assert len(pos_all) == 13
assert pos_all.most_common(1) == [("x", 2800)], "标点/分隔符是词性分布的第一名 —— 所以停用词表必须包含标点"
assert pos_all["v"] == 2081 and pos_all["n"] == 1372 and pos_all["eng"] == 1140

## 任务 12：TF-IDF / TextRank 关键词

In [ ]:
kw_tfidf = jieba.analyse.extract_tags(SAMPLE, topK=5)
kw_tr = jieba.analyse.textrank(SAMPLE, topK=5)

print("TF-IDF  :", kw_tfidf)
print("TextRank:", kw_tr)
print("交集    :", sorted(set(kw_tfidf) & set(kw_tr)))
print()
print("两者都用到「没被用户词典修正过的分词」，所以关键词里依然是「主变 / 渗漏」这种半截词。")

# ---- 验收 ----
assert kw_tfidf == ["主变", "母线", "渗漏", "巡检", "核实"]
assert kw_tr == ["渗漏", "巡检", "发现", "主变", "位于"]
assert set(kw_tfidf) & set(kw_tr) == {"主变", "巡检", "渗漏"}

## 任务 13：570 条量化对账

统计两边的词表大小、平均 token 数、总 token 数，以及「逐句完全一致」的条数。

In [ ]:
hand_tok = [[w for w in tokenize(t) if w.strip()] for t in texts]

t0 = time.time()
jb_tok = [jieba.lcut(t) for t in texts]
jb_sec = time.time() - t0

hand_vocab = len({w for d in hand_tok for w in d})
jb_vocab = len({w for d in jb_tok for w in d})
hand_avg = float(np.mean([len(d) for d in hand_tok]))
jb_avg = float(np.mean([len(d) for d in jb_tok]))
hand_total = sum(len(d) for d in hand_tok)
jb_total = sum(len(d) for d in jb_tok)
n_same = sum(1 for a, b in zip(hand_tok, jb_tok) if a == b)

print("手写  词表 %3d | 平均 %9.6f | 总 %5d" % (hand_vocab, hand_avg, hand_total))
print("jieba 词表 %3d | 平均 %9.6f | 总 %5d" % (jb_vocab, jb_avg, jb_total))
print("jieba 切完 %d 条耗时 %.3fs" % (len(texts), jb_sec))
print()
only_hand = sorted({w for d in hand_tok for w in d} - {w for d in jb_tok for w in d})
only_jb = sorted({w for d in jb_tok for w in d} - {w for d in hand_tok for w in d})
print("只在手写出现 %2d 个：" % len(only_hand), only_hand[:5], "...")
print("只在 jieba 出现 %2d 个：" % len(only_jb), only_jb[:5], "...")
print()
print("逐句完全一致：%d / %d = %.6f" % (n_same, len(hand_tok), n_same / len(hand_tok)))

# ---- 验收 ----
assert hand_vocab == 77 and jb_vocab == 94
assert abs(hand_avg - 8.536842) < 1e-6 and abs(jb_avg - 15.768421) < 1e-6
assert hand_total == 4866 and jb_total == 8988
assert len(only_hand) == 34 and len(only_jb) == 51
assert n_same == 0, "两种切法一句都对不上：手写把整句短语当词，jieba 把编号切散"
assert jb_sec < 5.0

In [ ]:
print("逐条对照（前 3 条）：")
for i in range(3):
    print()
    print(texts[i])
    print("  手写 :", "/".join(hand_tok[i]))
    print("  jieba:", "/".join(jb_tok[i]))

print()
print("两边各有各的错：")
print("  · 手写版把「请核实后反馈 / 请安排抢修」当成一个词 —— 那是词表里故意塞的长词；")
print("  · jieba 把台区编号 STATION_E_02 切成 STATION / _ / E / _ / 02，还把「镀锌层脱落」切成「镀锌」「层」「脱落」。")
print("  · 手写版在这批数据上反而更「像人」——因为它见过这个数据集，jieba 的通用词典没见过。")
print()
print("但反过来看 §8.9：只要给 jieba 一份用户词典，它立刻就能学会「母线巡检 / 3号主变 / 渗漏油」。")

## 任务 14：下游检验（切分方式 × 分类器）

三种切法喂同一个分类器。**答案会让你失望：全部打平。**
但请想清楚原因，这是本章最有价值的一条结论。

In [ ]:
def bow_cv5(docs):
    """词袋 + 逻辑回归 + 5 折交叉验证，返回 (词表大小, 特征维度, 平均准确率)。"""
    cv = CountVectorizer(tokenizer=lambda x: x, lowercase=False, token_pattern=None)
    M = cv.fit_transform(docs)
    scores = cross_val_score(LogisticRegression(max_iter=1000), M, labels, cv=5)
    return len(cv.vocabulary_), M.shape[1], float(scores.mean())


jd_tok = [tk.lcut(t) for t in texts]
rows_ds = [("手写最大匹配", bow_cv5(hand_tok)),
           ("jieba 默认", bow_cv5(jb_tok)),
           ("jieba + 用户词典", bow_cv5(jd_tok))]

print("三种切分 × 同一分类器（词袋 + 逻辑回归，5 折）：")
for tag, (n_vocab, n_feat, score) in rows_ds:
    print("  %-18s 词表 %3d | 特征 %3d | 5 折 %.6f" % (tag, n_vocab, n_feat, score))
print()
print("三种切法的词表大小差 22%（77 vs 94），准确率却一模一样。")
print("原因在 §2.3：这份数据的五类信号词**互不重叠**，分词方式动不了「渗油 / 锈蚀 / 碎裂」这些锚点。")

# ---- 验收 ----
assert [r[1][2] for r in rows_ds] == [1.0, 1.0, 1.0], "三种切法打平 —— 又一个负结果"
assert [r[1][1] for r in rows_ds] == [77, 94, 94]

## 自查清单

- [ ] 调用了 `jieba.initialize()`（不调的话 `dt.total` 是 0）
- [ ] 能说出 `freq==0` 的条目是什么（前缀，不是词）
- [ ] DAG 里 `frag in FREQ` 管「继续扩」，`FREQ[frag]` 管「算不算词」
- [ ] 知道 `jieba.dt.total != sum(FREQ.values())`
- [ ] 裸 DP 与 `jieba(HMM=False)` 一致率是 0/570
- [ ] 补上英数符合并后一致率是 570/570
- [ ] 全模式与搜索引擎模式的区别说得清（重叠 vs 补充细粒度）
- [ ] 用 `jieba.Tokenizer()` 而不是 `load_userdict` 改全局
- [ ] 用户词典里的词频给得足够大
- [ ] 举得出 HMM **改对**和**改错**各一个例子
- [ ] 知道全语料词性的第一名是 `x`（标点）
- [ ] 接受「三种切法下游 5 折准确率全是 1.000000」这个结果，并能解释原因